[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/python_build123d_basics/blob/master/docs/tuto_colab_build/build123d_multiplos_pav_perfil.ipynb)


# Exercício: Multiplos Pavimentos com importação de perfis
## build123d para Arquitetos e Engenheiros
### Versão Google Colab

Importando perfis do Autocad(dxf) e do rhino(step)



## Instalação

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "cadquery-simpleviewer[build123d,interactive]"],
        check=True,
    )
    # build123d pulls in a newer ipython than Colab's kernel bootstrap
    # tolerates. Put Colab's version back on disk — do NOT restart the
    # runtime, the current kernel already has the working ipython loaded.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "ipython==7.34.0", "--no-deps"],
        check=True,
    )

else:
    print("Not running in Colab, skipping package installation.")


## Importação dos pacotes

In [ ]:
import build123d as b3d
from cadquery_simpleviewer import show
import ezdxf

## Implementação

### Filtrando uma layer do DXF

O `import_dxf()` do build123d lê **todas** as entidades do modelspace de uma vez — diferente do `cq.importers.importDXF(..., include=[...])` do CadQuery, que já permite escolher a layer na importação. Como o arquivo `perfil_autocad.dxf` tem várias layers (`profile_01`, `profile_02`, ...), usamos o **ezdxf** (que o próprio build123d já usa por baixo dos panos para ler DXF) para copiar apenas as entidades da layer desejada para um arquivo temporário, e então deixamos o `import_dxf()` do build123d processar esse arquivo filtrado:

In [ ]:
def import_dxf_layer(path, layer):
    from pathlib import Path
    if not Path(path).exists():
        alt = Path("docs/tuto_colab_build") / path
        if alt.exists():
            path = str(alt)
    """Importa apenas as entidades de uma layer específica de um DXF."""
    doc = ezdxf.readfile(path)
    msp = doc.modelspace()

    filtered_doc = ezdxf.new(dxfversion=doc.dxfversion)
    filtered_msp = filtered_doc.modelspace()
    for entity in msp.query(f'*[layer=="{layer}"]'):
        filtered_msp.add_foreign_entity(entity)

    tmp_path = f"_layer_{layer}.dxf"
    filtered_doc.saveas(tmp_path)
    return b3d.import_dxf(tmp_path)

In [ ]:
# ── Parâmetros ──────────────────────────────────────────────────────────────
cota_inicial = 0
pap     = 3
n_pav   = 30


# ── Leitura do perfil ────────────────────────────────────────────────────────
perfil      = import_dxf_layer("perfil_autocad.dxf", "profile_01")
face_perfil = b3d.make_face(perfil)
# perfil_step = scale(import_step("perfil_step_214.step"), by=0.001)   # STEP from Rhino
# face_perfil = make_face(perfil_step.wires())

# ── Geração dos pavimentos ───────────────────────────────────────────────────
lista_pav = []

for i in range(n_pav + 1):
    cota_atual = round(cota_inicial + pap * i, 2)

    pav = (
        b3d.extrude(face_perfil, amount=pap)
        .translate((0, 0, cota_atual))
    )

    lista_pav.append(pav)
    # print(f"Cota do pavimento {i} = {cota_atual}")

show(lista_pav)


### Rotação incremental
copie o código acima e
adicione uma rotação gradual a cada andar

In [ ]:
# ── Parâmetros ──────────────────────────────────────────────────────────────
cota_inicial = 0
pap     = 3
n_pav   = 30
rot_inc = 2.5

# ── Leitura do perfil ────────────────────────────────────────────────────────
perfil      = import_dxf_layer("perfil_autocad.dxf", "profile_01")
face_perfil = b3d.make_face(perfil)
# perfil_step = scale(import_step("perfil_step_214.step"), by=0.001)   # STEP from Rhino
# face_perfil = make_face(perfil_step.wires())

# ── Geração dos pavimentos ───────────────────────────────────────────────────
lista_pav = []



for i in range(n_pav + 1):
    cota_atual = round(cota_inicial + pap * i, 2)

    pav = (
        b3d.extrude(face_perfil, amount=pap)
        .translate((0, 0, cota_atual))
        .rotate(b3d.Axis.Z, i * rot_inc)
    )

    lista_pav.append(pav)
    # print(f"Cota do pavimento {i} = {cota_atual}")

show(lista_pav)



### Variação com escala

Diferente do CadQuery — que precisa de um `cq.Matrix` e `.transformGeometry()` para escalar X/Y e transladar Z em uma única operação — o build123d já resolve isso com duas chamadas nativas simples: **`b3d.scale()`** (independente por eixo) seguida de **`.translate()`**.

In [ ]:
import numpy as np

# ── Parâmetros ──────────────────────────────────────────────────────────────
cota_inicial = 0
pap     = 3
n_pav   = 30
rot_inc = 2.5

smooth_factor = 1/(np.pi *1.5)

# ── Leitura do perfil ────────────────────────────────────────────────────────
perfil      = import_dxf_layer("perfil_autocad.dxf", "profile_01")
face_perfil = b3d.make_face(perfil)

# ── Geração dos pavimentos ───────────────────────────────────────────────────
lista_pav = []

for i in range(n_pav + 1):
    cota_atual = round(cota_inicial + pap * i, 2)
    scale_factor = np.abs(np.sin(i) * smooth_factor + 1)

    perfil_escalado = b3d.scale(face_perfil, by=(scale_factor, scale_factor, 1))

    pav = (
        b3d.extrude(perfil_escalado, amount=pap)
        .translate((0, 0, cota_atual))
        .rotate(b3d.Axis.Z, i * rot_inc)
    )

    lista_pav.append(pav)

show(lista_pav)

### Exportação

In [ ]:
# Atribui um label (nome) a cada pavimento e agrupa tudo em um Compound
for i, pav in enumerate(lista_pav):
    pav.label = f"pav_{i}"

assy = b3d.Compound(children=lista_pav)

# Exporta o Compound para um arquivo STEP
b3d.export_step(assy, "output.step")

# Alternativa: unir (fundir) os sólidos com "+" antes de exportar,
# equivalente ao mode="fused" do cq.Assembly.export() do CadQuery
# fused = lista_pav[0]
# for pav in lista_pav[1:]:
#     fused = fused + pav
# export_step(fused, "output.step")